# 4.2 정책 반복과 GridWorld 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter04_2_policy_iteration.ipynb)

책 본문: [4.2절](https://smhanlab.com/book-ml/kor/ml2/chapter04.html)

이 노트북은 책 4.2절의 코드(`policy_iteration`, `modified_policy_iteration`,
동점 실습)를 그대로 실행하고, "왼쪽" 출발이 바깥 반복마다 한 칸씩 고쳐지는
과정과 `k` sweep 수정 정책 반복의 sweep 수를 직접 확인합니다.


In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"


## 1. 5칸 GridWorld MDP 만들기

한 줄로 늘어선 5개의 칸(0~4). 왼쪽(0)/오른쪽(1)으로만 이동, 칸 4(목표)는
종료 상태(보상 0), 나머지 이동은 전부 보상 -1. `\gamma = 0.9`.
최단 경로로 목표에 도달하는 정책이 최적이 된다.


In [2]:
n_states, n_actions, gamma = 5, 2, 0.9  # action 0=left, 1=right
P, R = {}, {}
for s in range(n_states):
    P[s], R[s] = {}, {}
    for a in range(n_actions):
        if s == 4:
            P[s][a], R[s][a] = [(1.0, 4)], 0.0   # 목표 칸은 종료 상태
        else:
            ns = max(0, s-1) if a == 0 else min(4, s+1)
            P[s][a], R[s][a] = [(1.0, ns)], -1.0


## 2. 정책평가 (4.1절 함수 재사용)

`theta = 10^{-6}` 기준으로 `V^\pi`를 수렴시킨다. 이번 절의 sweep
카운팅을 위해 매 sweep의 최대 변화량 `delta`도 함께 반환한다.


In [3]:
def policy_evaluation(P, R, policy, gamma, theta=1e-6, count_sweeps=False):
    n_states = len(P)
    V = [0.0] * n_states
    sweeps = 0
    while True:
        delta = 0
        for s in range(n_states):
            a = policy[s]
            v_new = R[s][a] + gamma * sum(prob * V[ns] for prob, ns in P[s][a])
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        sweeps += 1
        if delta < theta:
            break
    return (V, sweeps) if count_sweeps else V


## 3. 정책 반복: "오른쪽"에서 시작하면 즉시 수렴

책 4.2절의 `policy_iteration`을 sweep/바깥 반복 카운터와 함께 실행한다.


In [4]:
def policy_iteration(P, R, n_actions, gamma, init_policy, count=False):
    n_states = len(P)
    policy = list(init_policy)
    outer, total_sweeps = 0, 0
    while True:
        outer += 1
        V, sweeps = policy_evaluation(P, R, policy, gamma, count_sweeps=True)
        total_sweeps += sweeps
        stable = True
        for s in range(n_states):
            best_a = max(range(n_actions),
                         key=lambda a: R[s][a] + gamma * sum(p * V[ns] for p, ns in P[s][a]))
            if best_a != policy[s]:
                policy[s] = best_a
                stable = False
        if count:
            print(f"outer {outer}: policy={[('L' if p == 0 else 'R') for p in policy[:4]] + ['·']}  "
                  f"평가 sweep {sweeps:3d}  V=[" + ", ".join(f"{v:8.2f}" for v in V) + "]")
        if stable:
            break
    if count:
        print(f"  -> outer {outer}회, 총 sweep {total_sweeps}")
    return V, policy

V, policy = policy_iteration(P, R, n_actions, gamma, [1] * n_states, count=True)
print("최종:", [f"{v:.4f}" for v in V], policy)


outer 1: policy=['R', 'R', 'R', 'R', '·']  평가 sweep   5  V=[   -3.44,    -2.71,    -1.90,    -1.00,     0.00]
outer 2: policy=['R', 'R', 'R', 'R', '·']  평가 sweep   5  V=[   -3.44,    -2.71,    -1.90,    -1.00,     0.00]
  -> outer 2회, 총 sweep 10
최종: ['-3.4390', '-2.7100', '-1.9000', '-1.0000', '0.0000'] [1, 1, 1, 1, 0]


## 4. "왼쪽"에서 시작하면 바깥 반복마다 한 칸씩

최악의 출발(모든 칸에서 왼쪽, 벽에 붙어 무한 루프 → `V = -10`)으로
돌리면, 정책이 **outer 1 → 칸 3, outer 2 → 칸 2, …** 순서로, 바깥
반복마다 **딱 한 칸씩** 오른쪽으로 고쳐진다. (목표 바로 옆인 칸 3에서
right가 가장 먼저 이기고, 그 정보가 한 칸씩 왼쪽으로 전파되기 때문.)


In [5]:
V, policy = policy_iteration(P, R, n_actions, gamma, [0] * n_states, count=True)
print("최종:", [f"{v:.4f}" for v in V], policy)


outer 1: policy=['L', 'L', 'L', 'R', '·']  평가 sweep 133  V=[  -10.00,   -10.00,   -10.00,   -10.00,     0.00]
outer 2: policy=['L', 'L', 'R', 'R', '·']  평가 sweep 133  V=[  -10.00,   -10.00,   -10.00,    -1.00,     0.00]
outer 3: policy=['L', 'R', 'R', 'R', '·']  평가 sweep 133  V=[  -10.00,   -10.00,    -1.90,    -1.00,     0.00]
outer 4: policy=['R', 'R', 'R', 'R', '·']  평가 sweep 133  V=[  -10.00,    -2.71,    -1.90,    -1.00,     0.00]
outer 5: policy=['R', 'R', 'R', 'R', '·']  평가 sweep   5  V=[   -3.44,    -2.71,    -1.90,    -1.00,     0.00]
  -> outer 5회, 총 sweep 537
최종: ['-3.4390', '-2.7100', '-1.9000', '-1.0000', '0.0000'] [1, 1, 1, 1, 0]


두 실행을 비교하면: "오른쪽" 출발은 outer 2회·총 sweep 10, "왼쪽"
출발은 outer 5회·총 sweep 537(= 133×4 + 5) — **같은 알고리즘인데
초기 정책만으로 총 계산량이 ~54배** 달라진다. 정책 반복의 속도는
초기 정책에 민감하다는 것의 실질이며, 4.3절의 가치 반복이 "초기
정책에 무관하게 `V*`로 직접 간다"는 설계 의도의 동기다.

그림으로 보면:


In [6]:
import numpy as np
states = np.arange(5)
# 목표는 칸 4: 최단거리 d = 4-s, V*(s) = -(1-gamma^d)/(1-gamma)
Vstar = [-(1 - gamma ** (4 - s)) / (1 - gamma) if s < 4 else 0.0 for s in states]

fig, ax = plt.subplots(figsize=(8, 3.6))
colors = ["#d9534f", "#f0ad4e", "#f0ad4e", "#5bc0de", "#5cb85c"]
ax.bar(states, Vstar, color=colors, width=0.62, zorder=3)
for s, v in zip(states, Vstar):
    ax.text(s, v - 0.18, f"{v:.3f}", ha="center", va="top", fontsize=10)
    ax.text(s, 0.22, r"$\rightarrow$" if s < 4 else "·",
            ha="center", va="bottom", fontsize=13)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(states)
ax.set_xticklabels([str(s) for s in states])
ax.set_xlabel("GridWorld 칸 (4 = 목표, 종료)", fontsize=11)
ax.set_ylabel(r"$V^*(s)$", fontsize=11)
ax.set_title(r"5칸 GridWorld의 최적 가치함수 $V^*$  (칸 0~3에서 $\pi^*(s) = \rightarrow$, 칸 4는 목표)",
             fontsize=12)
ax.set_ylim(-4.2, 0.9)
ax.grid(axis="y", ls="--", alpha=0.4, zorder=0)
fig.tight_layout()
fig.savefig(IMG + "/ch04_2_policy_iteration_walkthrough.svg", bbox_inches="tight")
plt.show()


## 5. 수정된 정책 반복: 가치 반복은 "k = 1"인 경우

정책평가를 끝까지 수렴시키지 않고 **k sweep만** 돌린 뒤 바로 개선하는
변형. `k = 1`이면 4.3절의 가치 반복과 정확히 같은 알고리즘이 된다.
(모두 "왼쪽" 출발, `theta` 판정 없이 "정책 변화 여부"로 멈춤.)


In [7]:
def modified_policy_iteration(P, R, n_actions, gamma, k, init_policy):
    n_states = len(P)
    policy = list(init_policy)
    V = [0.0] * n_states
    outer, total_sweeps = 0, 0
    while True:
        outer += 1
        for _ in range(k):                       # 평가: k sweep만
            for s in range(n_states):
                a = policy[s]
                V[s] = R[s][a] + gamma * sum(p * V[ns] for p, ns in P[s][a])
        total_sweeps += k
        stable = True
        for s in range(n_states):                # 개선: argmax
            best_a = max(range(n_actions),
                         key=lambda a: R[s][a] + gamma * sum(p * V[ns] for p, ns in P[s][a]))
            if best_a != policy[s]:
                policy[s] = best_a
                stable = False
        if stable:
            break
    return V, policy, outer, total_sweeps

print(f"{'k':>3} | {'바깥 반복':>6} | {'총 sweep':>6} | 최종 V")
for k in [1, 2, 3, 5, 10]:
    V, policy, outer, sweeps = modified_policy_iteration(P, R, n_actions, gamma, k, [0] * n_states)
    print(f"{k:>3} | {outer:>6} | {sweeps:>6} | " + " ".join(f"{v:7.3f}" for v in V))


  k |  바깥 반복 | 총 sweep | 최종 V
  1 |      5 |      5 |  -3.439  -2.710  -1.900  -1.000   0.000
  2 |      5 |     10 |  -3.439  -2.710  -1.900  -1.000   0.000
  3 |      5 |     15 |  -3.439  -2.710  -1.900  -1.000   0.000
  5 |      5 |     25 |  -3.439  -2.710  -1.900  -1.000   0.000
 10 |      5 |     50 |  -3.439  -2.710  -1.900  -1.000   0.000


## 6. 동점 실습: 정책은 다를 수 있으나 `V*`는 유일

상태 0이 두 개의 문(행동 0, 1)을 갖고 각각 방 1/방 2(보상 +1, 종료,
self-loop)로 가는 MDP. 상태 0에서 두 행동은 **완벽한 동점**
(`Q(0,·) = 1`). 동점 처리만 'first'와 'last'로 바꾸면:


In [8]:
n_states, n_actions, gamma = 3, 2, 0.9
P, R = {}, {}
for s in range(n_states):
    P[s], R[s] = {}, {}
P[0][0], R[0][0] = [(1.0, 1)], 1.0   # 문 a0 → 방 1 (+1, 종료)
P[0][1], R[0][1] = [(1.0, 2)], 1.0   # 문 a1 → 방 2 (+1, 종료)
P[1][0], R[1][0] = [(1.0, 1)], 0.0   # 방 1: self-loop (종료)
P[1][1], R[1][1] = [(1.0, 1)], 0.0
P[2][0], R[2][0] = [(1.0, 2)], 0.0   # 방 2: self-loop (종료)
P[2][1], R[2][1] = [(1.0, 2)], 0.0

def policy_iteration_tie(P, R, n_actions, gamma, tie_break):
    """tie_break: 'first' = 동점 시 가장 작은 행동 번호,
                   'last'  = 동점 시 가장 큰 행동 번호."""
    policy = [0] * n_states
    while True:
        V = policy_evaluation(P, R, policy, gamma)
        stable = True
        for s in range(n_states):
            qs = [R[s][a] + gamma * sum(p*V[ns] for p, ns in P[s][a])
                  for a in range(n_actions)]
            best = max(qs)
            cands = [a for a in range(n_actions) if abs(qs[a] - best) < 1e-12]
            chosen = cands[0] if tie_break == 'first' else cands[-1]
            if chosen != policy[s]:
                policy[s] = chosen
                stable = False
        if stable:
            return V, policy

V_first, pol_first = policy_iteration_tie(P, R, n_actions, gamma, 'first')
V_last,  pol_last  = policy_iteration_tie(P, R, n_actions, gamma, 'last')
print("first:", pol_first, [f"{v:.4f}" for v in V_first])
print("last :", pol_last,  [f"{v:.4f}" for v in V_last])
print("V* 일치:", all(abs(a - b) < 1e-9 for a, b in zip(V_first, V_last)))


first: [0, 0, 0] ['1.0000', '0.0000', '0.0000']
last : [1, 1, 1] ['1.0000', '0.0000', '0.0000']
V* 일치: True


## 요약

- **정책 반복** = 정책평가(4.1절) + 정책개선(각 행의 argmax)을 번갈아
  돌리는 것. "개선이 아무것도 바꾸지 않았다" = 벨만 최적방정식 성립 =
  최적.
- **단조 증가**: 모든 상태에서 동시에 개선하면 `V`는 감소하지 않는다.
- **수정된 정책 반복**(`k` sweep): `k=1`이 가치 반복, `k→∞`가
  완전 정책 반복. 세 방법 모두 같은 `V*`에 도달 — 다른 것은 속도.
- **동점**: 결과 *정책*은 동점 처리에 따라 달라져도, *가치 `V*`*는
  항상 유일하다.
